# Lesson 07: Model Fine-Tuning — Creating Your Own AI Expert

## Learning Objectives
- Understand the concept of fine-tuning: giving a general model "specialized training"
- Use code to simulate the effect of fine-tuning and compare it with the base model
- Understand the format and quality requirements of fine-tuning data
- Master the decision framework: when to fine-tune vs when not to

> Fine-tuning turns a general-purpose AI into your domain expert. But fine-tuning is not a silver bullet — knowing when to use it is key.

## Environment Setup

> Please run `00_Environment_Setup.ipynb` first to set up dependencies and API keys,
> then return to this notebook.

Once done, run the cell below to load environment variables:

In [ ]:
# Load API key from .env file (no need to enter it every time)
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ===== Pick a provider: change this one line, nothing else =====
#   'openai'     cloud  needs OPENAI_API_KEY      strongest, has embeddings
#   'deepseek'   cloud  needs DEEPSEEK_API_KEY    cheapest cloud, no embeddings
#   'openrouter' cloud  needs OPENROUTER_API_KEY  many vendors, no embeddings
#   'ollama'     local  no key, free and offline  run `ollama serve` and pull the model first
PROVIDER = 'openai'

# All four speak the OpenAI API format. They differ only in URL, key, model names.
PROVIDERS = {
    'openai': {
        'base_url': None,                            # None = OpenAI's default endpoint
        'api_key': os.getenv('OPENAI_API_KEY'),
        'model': 'gpt-5.6-luna',                     # small model: cheap and fast
        'model_big': 'gpt-5.6-terra',                # big model: pricier and stronger
        'embedding_model': 'text-embedding-3-small',
    },
    'deepseek': {
        'base_url': 'https://api.deepseek.com/v1',
        'api_key': os.getenv('DEEPSEEK_API_KEY'),
        'model': 'deepseek-v4-flash',                # fast and cheap
        'model_big': 'deepseek-v4-pro',              # stronger and slower; both V4 models think first
        'embedding_model': None,                     # DeepSeek has no embeddings endpoint
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'api_key': os.getenv('OPENROUTER_API_KEY'),
        'model': 'openai/gpt-5.6-luna',
        'model_big': 'openai/gpt-5.6-terra',
        'embedding_model': None,                     # OpenRouter does not proxy embeddings
    },
    'ollama': {
        'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
        'api_key': 'ollama',                         # local models ignore the key
        'model': 'gemma4:e2b-mlx',                   # run: ollama pull gemma4:e2b-mlx
        'model_big': 'gemma4:e2b-mlx',
        'embedding_model': 'nomic-embed-text',       # run: ollama pull nomic-embed-text
    },
}

cfg = PROVIDERS[PROVIDER]

# Check the key first: with no key, the OpenAI client raises a long traceback.
if not cfg['api_key']:
    raise SystemExit(
        f"No API key for '{PROVIDER}'. Either add {PROVIDER.upper()}_API_KEY to your .env file,\n"
        f"or set PROVIDER = 'ollama' above to run locally with no key at all."
    )

client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])

# Every lesson below uses only these three names, so switching provider needs no
# other code change.
MODEL = cfg['model']
MODEL_BIG = cfg['model_big']
EMBEDDING_MODEL = cfg['embedding_model']

print(f'Connected! provider = {PROVIDER}, default model = {MODEL}')


---

## Activity 1: Experience the Difference Between "General Model" and "Simulated Fine-Tuned Model"

### Activity Goal
Real fine-tuning requires significant GPU and time, making it unsuitable for classroom demos. However, we can use the system prompt to simulate the fine-tuning effect — injecting domain knowledge into the AI and observing how it shifts from generic answers to expert-level responses.

This is essentially prompt engineering, but it helps you intuitively understand what fine-tuning aims to achieve: making the model behave like an expert in a specific domain.

In [ ]:
# Activity 1: Simulate fine-tuning effect

# A highly specialized technical question
question = ('In a RAG system, how should you trade off between Chunk Size '
           '(document chunk size) and Top-K (number of retrieved chunks)?')

# Scenario A: General model (no domain knowledge injected)
print('=== Scenario A: General Model (No Domain Knowledge) ===')
r1 = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':question}],
    temperature=0.3)
generic_answer = r1.choices[0].message.content
print(generic_answer)

# Scenario B: "Simulated Fine-Tuning" — inject domain expertise
print('\n=== Scenario B: "Simulated Fine-Tuning" — Domain Expertise Injected ===')
domain_knowledge = (
    'You are a RAG system design expert with the following domain knowledge:\n'
    '1. Smaller Chunk Size improves retrieval precision but may lose context; recommended range is 256-1024 tokens\n'
    '2. Larger Top-K improves recall but introduces more noise; recommended range is 3-10\n'
    '3. Rule of thumb: factual questions need small Chunk + large Top-K; reasoning questions need large Chunk + small Top-K\n'
    '4. Recommended starting point: Chunk Size=512, Top-K=5, then adjust based on actual results\n'
    '5. If user feedback says answers are incomplete, increase Top-K; if answers are long and messy, decrease Top-K'
)
r2 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':domain_knowledge},
        {'role':'user','content':question}
    ],
    temperature=0.3)
fine_tuned_answer = r2.choices[0].message.content
print(fine_tuned_answer)

# Compare the two answers
print('\n=== AI Judge Comparison ===')
comparison = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':
        f'Compare the following two answers about RAG parameter trade-off. '
        f'Identify which one is more professional and insightful:\n\n'
        f'Answer A (General Model): {generic_answer}\n\n'
        f'Answer B (With Domain Expertise): {fine_tuned_answer}'
    }],
    temperature=0.2)
print(comparison.choices[0].message.content)

### Discussion
- How did the AI's response change after injecting domain knowledge?
- What is the fundamental difference between "simulated fine-tuning" and real fine-tuning?
  - Prompt engineering: you must repeat the knowledge injection in every prompt
  - Real fine-tuning: the knowledge is baked into the model parameters; you don't need to repeat it every time

---

## Activity 2: Fine-Tuning vs RAG vs Prompt Engineering — A Three-Way Comparison

### Activity Goal
These are three different approaches to adapting AI for specific scenarios. Let's compare their suitability for a concrete use case.

In [ ]:
# Activity 2: Three-way comparison of adaptation methods

scenario = ('You run a law firm and need an AI assistant to answer '
           'common questions from clients about employment law.')

comparison_prompt = f'''Scenario: {scenario}

Please analyze the suitability of each method across the following three dimensions:
1. Accuracy: How reliable are the answers?
2. Cost: Implementation and ongoing maintenance cost
3. Flexibility: When laws change, how easy is it to update?

Output the comparison in table format.'''

print('=== Three-Way Comparison Analysis ===')
r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'You are an AI Engineering consultant.'},
        {'role':'user','content':comparison_prompt}
    ],
    temperature=0.3)
print(r.choices[0].message.content)

# Follow-up: which approach do you recommend?
print('\n=== Recommendation ===')
r2 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'user','content':f'For the scenario: {scenario}, which approach would you recommend? Give specific reasons and an implementation plan.'}
    ],
    temperature=0.3)
print(r2.choices[0].message.content)

### Discussion
- Among the three methods, which is best suited for the "employment law Q&A" scenario?
- Under what circumstances is fine-tuning the best choice? When is RAG better?
- "Fine-tuning and RAG can be combined" — can you think of a scenario where both are used together?

---

## Activity 3: Design a Fine-Tuning Dataset

### Activity Goal
Fine-tuning requires "training data" — a set of high-quality Q&A pairs. This activity lets you experience the data preparation process.
Most people think the hardest part of fine-tuning is the technology. It's actually the data.

In [ ]:
# Activity 3: Design a fine-tuning dataset

# Scenario: Fine-tuning a "Coffee Shop Customer Service AI"
scenario_desc = ('You are the operations manager of a chain of coffee shops. '
                'You need to fine-tune a customer service AI. '
                'The AI should be able to answer questions about the menu, '
                'business hours, loyalty points, returns and exchanges, etc.')

print('Scenario:', scenario_desc)

# Let AI help you generate training data samples
data_gen_prompt = f'''{scenario_desc}

Please generate 10 "customer question → ideal customer service answer" training pairs, covering the following categories:
1. Menu inquiries (2 pairs)
2. Business hours (2 pairs)
3. Loyalty points (2 pairs)
4. Complaint handling (2 pairs)
5. Special requests (2 pairs)

Requirements:
- Answers should match the brand tone: warm, professional, efficient
- Answers should include specific information (no vague responses)
- Format: Q: ...\nA: ...'''

r = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':data_gen_prompt}],
    temperature=0.7)
print(r.choices[0].message.content)

print('\n---')
print('Checklist:')
print('1. Do these Q&A pairs cover all the scenarios you need?')
print('2. Does the tone of the answers fit the brand?')
print('3. Are there any vague or half-hearted answers that need revision?')
print('4. 10 pairs is far from enough — real fine-tuning typically requires hundreds to thousands of pairs!')

### Discussion
- How is the quality of the AI-generated training data? Would you need to manually correct anything?
- If this data contains bias or errors, what would happen to the fine-tuned model?
- Data quality > data quantity — do you agree?

---

## Lesson Review

| Skill | Description |
|-------|-------------|
| Fine-tuning concept | Understand that fine-tuning endows a general model with specialized skills |
| Three-way comparison | Understand the suitable scenarios for fine-tuning vs RAG vs prompt engineering |
| Dataset design | Experience the process and quality requirements of preparing fine-tuning data |

### Homework
1. Visit the OpenAI Platform fine-tuning page to learn about pricing and workflow
2. Search for "LoRA fine-tuning explained simply" to understand parameter-efficient fine-tuning
3. Think: if you were to fine-tune an AI, which scenario would you choose? How much data would you need?